In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os

# ==========================================
# 1. LOAD DATA
# ==========================================
print("Loading data...")
# Load your dataset
model_data = pd.read_parquet("../data/processed/preprocessed_data_enriched.parquet")
# Create "Time Dummy" directly on the main dataframe for Linear Regression
model_data["time_step"] = (model_data["date"] - model_data["date"].min()).dt.days

# ==========================================
# 2. DEFINE THE SPLIT MASKS (NO COPIES!)
# ==========================================
max_date = model_data["date"].max()
split_date = max_date - pd.Timedelta(days=28)

print(f"Dataset ranges from {model_data['date'].min().date()} to {max_date.date()}")
print(f"Validation starts on: {split_date.date()}")

train_mask = model_data["date"] < split_date
val_mask = model_data["date"] >= split_date

# ==========================================
# 3. STEP 1: LINEAR REGRESSION (The Trend)
# ==========================================
print("Training Linear Regression (Trend Baseline)...")
lr_model = LinearRegression()

# Fit only on the training slice using just the time step
lr_model.fit(
    model_data.loc[train_mask, ["time_step"]],
    model_data.loc[train_mask, "sales"]
)

# Predict baseline trend
lr_pred_train = lr_model.predict(model_data.loc[train_mask, ["time_step"]])
lr_pred_val = lr_model.predict(model_data.loc[val_mask, ["time_step"]])

# ==========================================
# 4. STEP 2: CALCULATE RESIDUALS (The Noise)
# ==========================================
# Actual Sales minus Trend Line = Residual Noise
y_train_resid = model_data.loc[train_mask, "sales"] - lr_pred_train
y_val_resid = model_data.loc[val_mask, "sales"] - lr_pred_val

# ==========================================
# 5. STEP 3: XGBOOST (The Pattern Finder)
# ==========================================
print("Training XGBoost on Residuals...")

drop_cols = ["id", "d", "date", "sales", "wm_yr_wk", "time_step"]
features = [col for col in model_data.columns if col not in drop_cols]

# Pull X slices on the fly to save RAM
X_train_xgb = model_data.loc[train_mask, features]
X_val_xgb = model_data.loc[val_mask, features]

print(f"Memory-optimized split & trend calculation complete! X_train_xgb shape: {X_train_xgb.shape}")

# Initialize XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    n_estimators=1500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50 # Stops training if validation doesn't improve for 50 rounds
)

# Fit the XGBoost model
xgb_model.fit(
    X_train_xgb, y_train_resid,
    eval_set=[(X_train_xgb, y_train_resid), (X_val_xgb, y_val_resid)],
    verbose=50 # Print progress every 50 trees
)

# ==========================================
# 6. STEP 4: THE HYBRID FORECAST (Trend + Noise)
# ==========================================
print("Calculating Final Hybrid Predictions...")

# XGBoost predicts the noise for the validation set
xgb_resid_pred = xgb_model.predict(X_val_xgb)

# Final Prediction = Linear Regression Trend + XGBoost Spikes
final_hybrid_pred = lr_pred_val + xgb_resid_pred

# Floor predictions at 0 (can't sell negative inventory)
final_hybrid_pred = np.clip(final_hybrid_pred, a_min=0, a_max=None)

# ==========================================
# 7. EVALUATION
# ==========================================
actual_val_sales = model_data.loc[val_mask, "sales"]

rmse = np.sqrt(mean_squared_error(actual_val_sales, final_hybrid_pred))
absolute_errors = np.abs(actual_val_sales - final_hybrid_pred)
wmape = np.sum(absolute_errors) / np.sum(actual_val_sales)
business_accuracy = (1 - wmape) * 100

print("\n--- Hybrid XGBoost Performance ---")
print(f"Validation RMSE:   {rmse:.4f}")
print(f"WMAPE:             {wmape:.4f} ({wmape * 100:.2f}%)")
print(f"Business Accuracy: {business_accuracy:.2f}%")

# ==========================================
# 8. SAVE MODELS
# ==========================================
os.makedirs("../models", exist_ok=True)
joblib.dump(lr_model, "../models/lr_trend_model_xgb.pkl")
joblib.dump(xgb_model, "../models/xgboost_residual_model.pkl")
print("\nSaved both LR and XGBoost models successfully!")

Loading data...
Dataset ranges from 2011-02-26 to 2016-04-24
Validation starts on: 2016-03-27
Training Linear Regression (Trend Baseline)...
Training XGBoost on Residuals...
Memory-optimized split & trend calculation complete! X_train_xgb shape: (4529102, 82)
[0]	validation_0-rmse:4.31486	validation_1-rmse:3.39051
[50]	validation_0-rmse:2.32055	validation_1-rmse:2.02546
[100]	validation_0-rmse:2.24758	validation_1-rmse:2.00987
[150]	validation_0-rmse:2.22189	validation_1-rmse:2.00605
[200]	validation_0-rmse:2.19970	validation_1-rmse:2.00567
[250]	validation_0-rmse:2.17918	validation_1-rmse:2.00505
[300]	validation_0-rmse:2.15997	validation_1-rmse:2.00216
[350]	validation_0-rmse:2.13999	validation_1-rmse:1.99845
[400]	validation_0-rmse:2.12269	validation_1-rmse:1.99822
[429]	validation_0-rmse:2.11416	validation_1-rmse:1.99861
Calculating Final Hybrid Predictions...

--- Hybrid XGBoost Performance ---
Validation RMSE:   1.9978
WMAPE:             0.6808 (68.08%)
Business Accuracy: 31.92%
